## module 1 — baseline classifier

plain resnet50 fine-tune, **kept fixed as the comparison point** — do not add upgrades here, they go in `04_train_improved`. you only customize the *augmentation* and *model / optimizer* cells; the plumbing lives in `bvtrain/`.

run-only: execute top-to-bottom to (re)produce the baseline checkpoint.

In [ ]:
# get bvtrain (shared training plumbing): locally it's ../bvtrain; on colab/kaggle we clone the repo
import os, sys
_CANDS = ["..", ".", "botanical-vision"]
if not any(os.path.isdir(f"{p}/bvtrain") for p in _CANDS):
    os.system("git clone -q https://github.com/babnigg/botanical-vision.git")
for _p in _CANDS:
    if os.path.isdir(f"{_p}/bvtrain"):
        sys.path.insert(0, _p)
        break

import torch
import torch.nn as nn
from torchvision import models, transforms
import bvtrain as bv

In [ ]:
env = bv.setup()

In [ ]:
N_SPECIES = None   # int (e.g. 100) for a quick subset run, None for all 4,094 species
data = bv.load_data(env, n_species=N_SPECIES)

In [ ]:
# ── customize: augmentation (light, baseline) ──
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(bv.MEAN, bv.STD),
])

In [ ]:
# ── customize: model, optimizer (plain baseline — no scheduler) ──
EPOCHS = 5
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, data.n_labels)
model = model.to(env.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
loaders = bv.build_loaders(data, train_tf, env)
hist = bv.fit(model, optimizer, loaders, epochs=EPOCHS, run_name="resnet50_baseline", env=env)

In [ ]:
bv.plot_history(hist)
bv.evaluate(model, loaders.test, env, run_name="resnet50_baseline")